# Bölüm 19 – Reinforcement Learning (Pekiştirmeli Öğrenme)

> **Türkçe Açıklamalı Notebook**  
> Orijinal kaynak: *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (Aurélien Géron)  
> Bu notebook, bölüm 19'daki tüm örnek kodları açıklamalarıyla birlikte içermektedir.

---

## Bu Notebook'ta Ne Öğreneceğiz?

| Konu | Açıklama |
|------|----------|
| **Gymnasium** | OpenAI'nin RL ortam kütüphanesi — CartPole, Atari vb. |
| **Hard-Coded Policy** | Elle yazılmış basit kontrol politikası |
| **Policy Network** | Sinir ağı tabanlı politika |
| **REINFORCE** | Policy gradient yöntemi |
| **Markov Chains** | Markov Zincirleri — stokastik durum geçişleri |
| **Markov Decision Process (MDP)** | Karar süreçleri, ödül, geçiş olasılıkları |
| **Q-Value Iteration** | Dinamik programlama ile optimal Q-değerleri |
| **Q-Learning** | Model-free Q-değeri öğrenimi |
| **Deep Q-Network (DQN)** | Derin sinir ağı ile Q-öğrenimi + replay buffer |
| **Actor-Critic** | Actor (politika) + Critic (değer) birlikte eğitim |
| **PPO (Proximal Policy Optimization)** | Stable-Baselines3 ile Atari Breakout oyunu |

---

## Temel Kavramlar

```
Agent (Ajan) ──action──▶ Environment (Ortam)
     ▲                        │
     │                        ▼
     └────observation, reward──┘
```

- **Agent**: Karar veren öğrenen sistem
- **Environment**: Ajanın etkileşime girdiği ortam
- **Observation (obs)**: Ortamın mevcut durumu
- **Action**: Ajanın yaptığı seçim
- **Reward**: Ajanın aldığı anlık ödül (pozitif veya negatif)
- **Episode**: Baştan sona bir oyun/deney
- **Policy (π)**: Gözlemden eyleme eşleme fonksiyonu


# 1. Kurulum (Setup)

## 1.1 Python Sürüm Kontrolü

Bu proje **Python 3.10 veya üzerini** gerektirir.

In [ ]:
import sys

# Python sürümünün 3.10 veya üzeri olduğunu kontrol et
# Daha eski bir sürümde AssertionError alırsınız
assert sys.version_info >= (3, 10)

## 1.2 Çalışma Ortamı Tespiti

In [ ]:
# Çalışma ortamını tespit et (Colab / Kaggle / yerel makine)
IS_COLAB = "google.colab" in sys.modules   # Google Colab ise True
IS_KAGGLE = "kaggle_secrets" in sys.modules  # Kaggle ise True

## 1.3 Gerekli Kütüphanelerin Kurulumu

Bu bölüm için iki önemli kütüphane gerekmektedir:

- **Box2D**: Gymnasium'un fizik tabanlı ortamları (LunarLander, BipedalWalker vb.) için gerekli 2D fizik motoru
- **stable_baselines3**: PPO, A2C, SAC gibi hazır RL algoritmalarını içeren kütüphane

Eski `gym` kütüphanesi kaldırılıp yerine `gymnasium` kurulmaktadır.


In [ ]:
# Colab'da eski gym kütüphanesini kaldır ve gerekli paketleri kur
if IS_COLAB:
    %pip uninstall -qy gym          # Eski gym'i kaldır (uyumsuzluk olabilir)
    %pip install -qU Box2D stable_baselines3  # Yeni paketleri kur

# Kaggle'da da aynı işlemi yap + gymnasium'u da güncelle
if IS_KAGGLE:
    %pip uninstall -qy gym
    %pip install -qU gymnasium
    %pip install -qU Box2D stable_baselines3

## 1.4 PyTorch Sürüm Kontrolü

In [ ]:
from packaging.version import Version
import torch

# PyTorch sürümünün 2.6.0 veya üzeri olduğunu kontrol et
assert Version(torch.__version__) >= Version("2.6.0")

## 1.5 Donanım Hızlandırıcı (Device) Seçimi

In [ ]:
# Kullanılabilir donanım hızlandırıcıya göre device belirle
# Sıralama: CUDA (NVIDIA GPU) > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = "cuda"          # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = "mps"           # Apple Silicon GPU
else:
    device = "cpu"           # GPU yoksa CPU

device

## 1.6 GPU Yoksa Uyarı

In [ ]:
# GPU bulunamazsa kullanıcıyı uyar ve nasıl etkinleştirileceğini açıkla
if device == "cpu":
    print("Neural nets can be very slow without a hardware accelerator.")
    if IS_COLAB:
        print("Go to Runtime > Change runtime and select a GPU hardware accelerator.")
    if IS_KAGGLE:
        print("Go to Settings > Accelerator and select GPU.")

## 1.7 Matplotlib Grafik Ayarları

Animasyon desteği için `plt.rc('animation', html='jshtml')` ekliyoruz. Bu, Jupyter Notebook içinde animasyonların HTML5 video olarak görüntülenmesini sağlar.

In [ ]:
import matplotlib.animation  # Animasyon oluşturmak için
import matplotlib.pyplot as plt

# Grafik yazı tipi boyutları
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

# Animasyonların Jupyter Notebook'ta HTML5 video olarak gösterilmesi için
# 'jshtml': JavaScript tabanlı HTML animasyon formatı
plt.rc('animation', html='jshtml')

## 1.8 TensorBoard Uzantısını Yükle

**TensorBoard**, eğitim sürecini gerçek zamanlı izlemeye yarayan bir görselleştirme aracıdır. PPO bölümünde eğitim metriklerini takip etmek için kullanılacak.

In [ ]:
# TensorBoard Jupyter uzantısını yükle
# Bu sayede %tensorboard magic komutu kullanılabilir
%load_ext tensorboard

# 2. Gymnasium'a Giriş

**Gymnasium** (eski adı: OpenAI Gym), Reinforcement Learning araştırması için standart ortam kütüphanesidir. İçinde:
- Klasik kontrol problemleri (CartPole, MountainCar, Pendulum)
- Atari oyunları (Breakout, Pong, Space Invaders)
- Robotik simülasyonlar (HalfCheetah, Ant, Hopper)
- Box2D fizik ortamları (LunarLander, BipedalWalker, CarRacing)

## CartPole-v1 Ortamı

Bir direksiyona (cart) monte edilmiş bir sırığı (pole) dengede tutma problemi.

```
      |
      | ← pole (sırık)
    __|__
   |cart | ← sola veya sağa iter
===|=====|===
```

**Observation Space**: [cart pozisyonu, cart hızı, pole açısı, pole açı hızı]  
**Action Space**: 0 (sola it) veya 1 (sağa it)  
**Reward**: Her adımda +1 (sırık devrilmediği sürece)  
**Terminal**: Sırık açısı ±12° veya cart ±2.4 birim dışına çıkarsa oyun biter


In [ ]:
import gymnasium as gym

# CartPole-v1 ortamını oluştur
# render_mode="rgb_array": Ortamın görsel görüntüsünü numpy array olarak döndür
# max_episode_steps=1000: Bir episode maksimum 1000 adımda kesilir (truncated)
env = gym.make("CartPole-v1", render_mode="rgb_array", max_episode_steps=1000)

In [ ]:
# Kayıtlı ortamların ilk 5'ini göster
# gym.envs.registry: Tüm kayıtlı Gymnasium ortamlarının sözlüğü
envs = gym.envs.registry
sorted(envs.keys())[:5] + ["..."]

In [ ]:
# CartPole-v1 ortamının teknik spesifikasyonunu göster
# Observation space, action space, reward threshold vb. bilgileri içerir
envs["CartPole-v1"]

In [ ]:
# Tüm kayıtlı Gymnasium ortamlarını kategorilere göre yazdır
# Çok sayıda ortam var: Atari, Box2D, Classic Control, MuJoCo vb.
gym.pprint_registry()

In [ ]:
# Ortamı sıfırla ve ilk gözlemi al
# seed=42: Tekrarlanabilirlik için rastgelelik tohumu
# Döndürür: (obs, info)
# obs: [cart_pos, cart_vel, pole_angle, pole_angular_vel]
obs, info = env.reset(seed=42)
obs  # İlk gözlem: 4 boyutlu float array

In [ ]:
# Sıfırlama sonrasında dönen info sözlüğü
# CartPole için genellikle boş, bazı ortamlarda ek bilgi içerebilir
info

In [ ]:
# Ortamı görsel olarak render et (RGB görüntü olarak)
# render_mode="rgb_array" ile kurulduğu için numpy array döner
img = env.render()
img.shape  # (yükseklik, genişlik, kanal_sayısı) — 3 kanal = R, G, B

In [ ]:
# Ortamı görselleştiren yardımcı fonksiyon
def plot_environment(env, figsize=(5, 4)):
    """Ortamın mevcut durumunu ekranda gösterir."""
    plt.figure(figsize=figsize)
    img = env.render()        # RGB array al
    plt.imshow(img)           # Göster
    plt.axis("off")           # Eksen numaralarını gizle
    return img

plot_environment(env)
plt.show()

In [ ]:
# Action space'i incele
# Discrete(2): 0 veya 1 olmak üzere 2 ayrık aksiyon var
# 0 = sola it, 1 = sağa it
env.action_space

In [ ]:
# Bir aksiyon uygula ve sonuçları gözlemle
action = 1  # Sağa it (arabayı sağa doğru ivmelendır)

# env.step(action) döndürür:
# obs: Yeni gözlem (yeni durum)
# reward: Anlık ödül
# done: Episode gerçekten bitti mi? (sırık düştü, sınır aşıldı)
# truncated: Episode maksimum adımda kesildi mi? (1000 adım)
# info: Ek bilgi sözlüğü
obs, reward, done, truncated, info = env.step(action)
obs  # Yeni gözlem

In [ ]:
# Aksiyon sonuçlarını göster
# reward=1.0: Her hayatta kalınan adım için +1 ödül
# done=False: Sırık henüz düşmedi
# truncated=False: 1000 adım dolmadı
reward, done, truncated, info

In [ ]:
# Episode bittiyse ortamı sıfırla
# done: Sırık çok eğildi veya araba sınır dışına çıktı
# truncated: Maksimum adım sayısına ulaşıldı
if done or truncated:
    obs, info = env.reset()

# 3. Basit Hard-Coded (Elle Yazılmış) Politika

**Policy (Politika)**: Gözlemden eyleme eşleme fonksiyonu — `π: obs → action`

En basit yaklaşım, fiziksel sezgiye dayalı elle yazılmış bir kural:
- Sırık sola eğiliyorsa → arabayı sola it (action=0)
- Sırık sağa eğiliyorsa → arabayı sağa it (action=1)

Bu, basit ve anlaşılır ama optimal değil.


In [ ]:
def basic_policy(obs):
    """
    Sırığın açısına göre basit bir kural tabanlı politika.
    
    Args:
        obs: [cart_pos, cart_vel, pole_angle, pole_angular_vel]
             obs[2] = pole_angle (sırık açısı, radyan cinsinden)
    
    Returns:
        0: Sol (sırık sola eğiliyorsa arabayı sola it)
        1: Sağ (sırık sağa eğiliyorsa arabayı sağa it)
    """
    angle = obs[2]          # Sırık açısını al (obs[2])
    return 0 if angle < 0 else 1  # Negatif açı = sola eğik → sola git

# 500 episode boyunca bu politikayı test et ve toplam ödülleri kaydet
totals = []
for episode in range(500):
    total_rewards = 0
    obs, info = env.reset(seed=episode)  # Her episode için farklı seed
    
    while True:  # Sonsuz döngü: done/truncated ile kesilir
        action = basic_policy(obs)           # Politikayı çalıştır
        obs, reward, done, truncated, info = env.step(action)
        total_rewards += reward              # Ödülü biriktir
        if done or truncated:
            break
    
    totals.append(total_rewards)

In [ ]:
import numpy as np

# 500 episode'un istatistiksel özeti
np.mean(totals), np.std(totals), min(totals), max(totals)
# Ortalama ~40-50 adım: Basit politika kararsız, sırık hızla düşüyor
# Maksimum 1000 gibi uç değerler nadiren görülür

## 3.1 Episode Animasyonu Gösterme

Eğitim ve test sonuçlarını görselleştirmek için animasyon fonksiyonları tanımlıyoruz.

In [ ]:
def update_scene(num, frames, patch):
    """
    Matplotlib FuncAnimation için her frame'de çağrılan güncelleme fonksiyonu.
    patch: matplotlib AxesImage nesnesi (imshow ile oluşturulmuş)
    num: Mevcut frame indeksi
    """
    patch.set_data(frames[num])  # Frame'i güncelle
    return patch,                # Tuple döndür (FuncAnimation gereksinimi)


def plot_animation(frames, repeat=False, interval=40):
    """
    Bir frame listesinden Matplotlib animasyonu oluşturur.
    
    Args:
        frames: Her adımın RGB görüntüsünü içeren liste
        repeat: Animasyonu döngüde oynat mı?
        interval: Frameler arası bekleme süresi (ms)
    
    Returns:
        FuncAnimation nesnesi (Jupyter'de HTML5 video olarak gösterilir)
    """
    fig = plt.figure()
    patch = plt.imshow(frames[0])   # İlk frame'i göster
    plt.axis('off')
    anim = matplotlib.animation.FuncAnimation(
        fig, update_scene, fargs=(frames, patch),
        frames=len(frames), repeat=repeat, interval=interval)
    plt.close()  # Statik grafik oluşmasını engelle
    return anim  # Animasyon nesnesini döndür


def show_one_episode(policy, seed=42):
    """
    Bir politika fonksiyonu alır, tam bir episode çalıştırır ve animasyon döndürür.
    
    Args:
        policy: obs → action fonksiyonu
        seed: Tekrarlanabilirlik için seed
    """
    frames = []  # Her adımdaki görüntüleri kaydet
    env = gym.make("CartPole-v1", render_mode="rgb_array",
                   max_episode_steps=1000)
    obs, info = env.reset(seed=seed)
    
    while True:
        frames.append(env.render())          # Mevcut kareyi kaydet
        action = policy(obs)                  # Politikadan aksiyon al
        obs, reward, done, truncated, info = env.step(action)
        if done or truncated:
            break
    
    env.close()                               # Ortam kaynaklarını serbest bırak
    return plot_animation(frames)             # Animasyon oluştur

# Basit politikanın animasyonunu göster
show_one_episode(basic_policy)
# Sonuç: Sırık çabuk devrilecek — politika çok ilkel

# 4. Sinir Ağı Politikası (Neural Network Policy)

Kural tabanlı politika yerine, **gözlemi girdi alarak en iyi eylemi tahmin eden** bir sinir ağı kullanıyoruz.

## Mimari

```
obs (4 boyut) → [Linear(4→5)] → ReLU → [Linear(5→1)] → logit
```

**Çıkış**: Tek bir logit değeri — sağa gitmek için ne kadar "eğilim" var?  
**Bernoulli Dağılımı**: Bu logit değerinden olasılık üretip örnekleme yapıyoruz.

## Neden Bernoulli?

CartPole sadece 2 aksiyon içerdiğinden (sola/sağa), **Bernoulli** (0/1) dağılımı yeterlidir.  
Daha fazla aksiyon olsaydı `Categorical` dağılımı kullanılırdı.


In [ ]:
import torch
import torch.nn as nn

class PolicyNetwork(nn.Module):
    """
    CartPole için sinir ağı tabanlı politika.
    
    Giriş: 4 boyutlu gözlem [cart_pos, cart_vel, pole_angle, pole_angular_vel]
    Çıkış: 1 boyutlu logit (sağa gitme eğilimi)
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 5),   # 4 gözlem → 5 gizli nöron
            nn.ReLU(),         # Non-linear aktivasyon
            nn.Linear(5, 1)    # 5 gizli → 1 logit çıkışı
        )

    def forward(self, state):
        """state: [4] boyutlu gözlem tensoru → logit: skaler"""
        return self.net(state)

In [ ]:
def choose_action(model, obs):
    """
    Politika modelinden bir aksiyon örnekler ve log olasılığını döndürür.
    
    Args:
        model: PolicyNetwork — gözlemi logit'e çevirir
        obs: NumPy array [cart_pos, cart_vel, pole_angle, pole_angular_vel]
    
    Returns:
        action: 0 veya 1 (int)
        log_prob: Seçilen aksiyonun log olasılığı (gradient hesabı için)
    
    Önemli: log_prob, REINFORCE algoritmasında policy gradient hesabında kullanılır.
    """
    state = torch.as_tensor(obs)     # NumPy → PyTorch tensor (kopyalamadan)
    logit = model(state)             # Sinir ağından logit al
    
    # Bernoulli dağılımı: logit'ten 0/1 örneklemesi
    # logits parametresi: sigmoid uygulamadan önceki ham değer
    # P(action=1) = sigmoid(logit)
    dist = torch.distributions.Bernoulli(logits=logit)
    
    action = dist.sample()              # Stokastik örnekleme: 0 veya 1
    log_prob = dist.log_prob(action)    # log π(a|s) — gradient için gerekli
    
    return int(action.item()), log_prob

In [ ]:
# Rastgele başlatılmış (henüz eğitilmemiş) politikanın animasyonunu göster
torch.manual_seed(42)
model = PolicyNetwork()
model.eval()  # Değerlendirme modu (Dropout/BatchNorm için önemli)

def neural_net_policy(obs):
    """PolicyNetwork'ü kullanan politika wrapper fonksiyonu."""
    with torch.no_grad():  # Gradient hesaplamayı kapat (değerlendirme)
        action, _ = choose_action(model, obs)
        return action

# Eğitilmemiş model rastgele hareket eder — sırık hemen devrilir
show_one_episode(neural_net_policy)

# 5. Policy Gradients — REINFORCE Algoritması

## Policy Gradient Temel Fikri

Bir politikanın ne kadar iyi olduğunu **Return (kümülatif ödül)** ile ölçeriz.  
Gradyan yükseltme ile politikayı iyileştiririz: iyi sonuçlara yol açan aksiyonların olasılığını artır, kötü sonuçlara yol açanların olasılığını azalt.

## REINFORCE Loss Fonksiyonu

```
Loss = -Σ_t [ log π(a_t|s_t) * G_t ]
```

Burada:
- `π(a_t|s_t)`: t adımında s_t durumundayken a_t aksiyonunu seçme olasılığı
- `G_t`: t adımından itibaren diskontlanmış kümülatif ödül (Return)

**Eksi işareti**: PyTorch gradient iner (minimize), biz maximize etmek istiyoruz.

## Discounted Return (Diskontlanmış Getiri)

```
G_t = r_t + γ*r_{t+1} + γ²*r_{t+2} + ... + γ^{T-t}*r_T
```

γ (discount factor) ∈ [0,1]:
- γ=0: Sadece anlık ödüle bak (açgözlü)
- γ=1: Tüm gelecek ödülleri eşit ağırlıkta say
- γ=0.95: Yakın geleceği tercih et, uzak geleceği hafifçe iskonto et


In [ ]:
def compute_returns(rewards, discount_factor):
    """
    Bir episode'un her adımı için diskontlanmış getiriyi (Return) hesaplar.
    
    G_t = r_t + γ*r_{t+1} + γ²*r_{t+2} + ...
    
    Geri yönlü hesaplama (verimli): Son adımdan başa doğru git
    returns[t-1] += returns[t] * gamma
    
    Örnek: rewards=[10, 0, -50], gamma=0.8
    - returns[2] = -50
    - returns[1] = 0 + (-50)*0.8 = -40
    - returns[0] = 10 + (-40)*0.8 = -22
    
    Args:
        rewards: Episode boyunca alınan ödüllerin listesi
        discount_factor: γ — gelecekteki ödüllerin iskonto oranı
    
    Returns:
        Her adım için diskontlanmış getiri (torch.tensor)
    """
    returns = rewards[:]  # Listeyi kopyala (orijinali değiştirme)
    
    # Sondan başa doğru geri yayılım ile hesapla
    for step in range(len(returns) - 1, 0, -1):
        returns[step - 1] += returns[step] * discount_factor
    
    return torch.tensor(returns)

In [ ]:
# Test: [10, 0, -50] ödülleri için discount_factor=0.8 ile return hesapla
compute_returns([10, 0, -50], discount_factor=0.8)
# Beklenen: [-22, -40, -50]
# -50 → -50 (son adım, iskonto yok)
# -40 → 0 + (-50)*0.8 = -40
# -22 → 10 + (-40)*0.8 = -22

In [ ]:
def run_episode(model, env, seed=None):
    """
    Bir tam episode çalıştırır ve log olasılıkları ile ödülleri toplar.
    
    Eğitim için:
    - log_probs: Her adımda seçilen aksiyonun log olasılığı (gradient için)
    - rewards: Her adımın anlık ödülü (return hesabı için)
    
    Args:
        model: PolicyNetwork
        env: Gymnasium ortamı
        seed: Tekrarlanabilirlik seed'i
    
    Returns:
        log_probs: log π(a_t|s_t) listesi
        rewards: r_t listesi
    """
    log_probs, rewards = [], []
    obs, info = env.reset(seed=seed)
    
    while True:  # Ortam max_episode_steps'e ulaşınca truncate eder
        action, log_prob = choose_action(model, obs)  # Politikadan aksiyon
        obs, reward, done, truncated, _info = env.step(action)
        
        log_probs.append(log_prob)   # Log olasılığı kaydet
        rewards.append(reward)       # Ödülü kaydet
        
        if done or truncated:
            return log_probs, rewards

In [ ]:
def train_reinforce(model, optimizer, env, n_episodes, discount_factor):
    """
    REINFORCE (Monte Carlo Policy Gradient) algoritması.
    
    Her episode sonunda:
    1. Tüm adımların return'lerini hesapla
    2. Return'leri normalize et (sıfır ortalama, birim varyans)
    3. Policy gradient loss'u hesapla: -Σ log π(a_t|s_t) * G_t
    4. Backpropagation ile politikayı güncelle
    
    Normalize etme neden önemli?
    - Farklı uzunluktaki episode'lar karşılaştırılabilir hale gelir
    - Gradient'lerin ölçeği sabit tutulur → daha kararlı eğitim
    - Baseline etkisi: Ortalama return'ü çıkarma varyansı azaltır
    
    Args:
        model: PolicyNetwork
        optimizer: Optimizasyon algoritması
        env: Gymnasium ortamı
        n_episodes: Kaç episode eğitilecek
        discount_factor: γ — gelecek ödüllerin iskonto oranı
    """
    for episode in range(n_episodes):
        # Rastgele seed: Her episode farklı ama tekrarlanabilir
        seed = torch.randint(0, 2**32, size=()).item()
        
        # Tam episode çalıştır → log olasılıklar ve ödüller
        log_probs, rewards = run_episode(model, env, seed=seed)
        
        # Her adım için diskontlanmış return hesapla
        returns = compute_returns(rewards, discount_factor)
        
        # Return'leri normalize et: (G_t - mean(G)) / std(G)
        # 1e-7: Sıfır standart sapma durumunda sıfıra bölmeyi önler
        std_returns = (returns - returns.mean()) / (returns.std() + 1e-7)
        
        # Policy gradient loss: -log π(a_t|s_t) * G_t toplamı
        # Eksi işareti: PyTorch minimize eder, biz maximize etmek istiyoruz
        losses = [-logp * rt for logp, rt in zip(log_probs, std_returns)]
        
        # Tüm adımların kayıplarını tek tensor'da birleştir ve topla
        loss = torch.cat(losses).sum()
        
        # Gradient hesapla ve parametreleri güncelle
        optimizer.zero_grad()   # Önceki gradient'leri sıfırla
        loss.backward()         # Geri yayılım
        optimizer.step()        # Parametreleri güncelle
        
        print(f"\rEpisode {episode + 1}, Reward: {sum(rewards):.2f}", end=" ")

In [ ]:
# REINFORCE ile PolicyNetwork'ü eğit
torch.manual_seed(42)
model = PolicyNetwork()
optimizer = torch.optim.NAdam(model.parameters(), lr=0.06)

# 200 episode, discount_factor=0.95
# NAdam: Adam + Nesterov momentum — policy gradient için iyi çalışır
train_reinforce(model, optimizer, env, n_episodes=200, discount_factor=0.95)

In [ ]:
# Eğitilmiş modelin performansını animasyonla göster
show_one_episode(neural_net_policy)
# Beklenti: Sırık çok daha uzun süre dengede kalmalı!

# 6. Ekstra Materyal — Markov Zincirleri (Markov Chains)

## Markov Özelliği

> "Gelecek yalnızca şimdiki duruma bağlıdır, geçmiş durumlar önemli değildir."

```
P(s_{t+1} | s_t, s_{t-1}, ..., s_0) = P(s_{t+1} | s_t)
```

Bu, birçok RL probleminin temel varsayımıdır.

## Geçiş Matrisi

`T[s, s']` = s durumundan s' durumuna geçiş olasılığı

Aşağıdaki örnekte 4 durum var (s0, s1, s2, s3):
- s3: Terminal durum (oyun bitti)
- Zincir s0'dan başlıyor


In [ ]:
np.random.seed(42)

# Markov Zinciri geçiş olasılık matrisi
# transition_probabilities[i][j] = s_i'den s_j'ye geçiş olasılığı
# Her satır toplamı 1.0 olmalı (olasılık dağılımı)
transition_probabilities = [
    # s0'dan: %70 s0'a kal, %20 s1'e git, %0 s2'ye, %10 s3'e
    [0.7, 0.2, 0.0, 0.1],
    # s1'den: %0 s0'a, %0 s1'e, %90 s2'ye, %10 s3'e
    [0.0, 0.0, 0.9, 0.1],
    # s2'den: %0 s0'a, %100 s1'e, %0 s2'ye, %0 s3'e
    [0.0, 1.0, 0.0, 0.0],
    # s3'ten: Terminal durum — %100 s3'te kal
    [0.0, 0.0, 0.0, 1.0]
]

n_max_steps = 1000   # Sonsuz döngüyü önlemek için maksimum adım sayısı
terminal_states = [3]  # s3 terminal durum

def run_chain(start_state):
    """
    Markov zincirini verilen başlangıç durumundan terminal duruma kadar çalıştırır.
    Her adımda mevcut durumu ekrana yazar.
    """
    current_state = start_state
    for step in range(n_max_steps):
        print(current_state, end=" ")
        if current_state in terminal_states:
            break
        # Geçiş olasılıklarına göre rastgele sonraki durumu seç
        current_state = np.random.choice(
            range(len(transition_probabilities)),
            p=transition_probabilities[current_state]
        )
    else:
        print("...", end="")  # 1000 adım doldu ama terminal'e ulaşılamadı
    print()

# 10 farklı run çalıştır (hepsi s0'dan başlıyor)
for idx in range(10):
    print(f"Run #{idx + 1}: ", end="")
    run_chain(start_state=0)
# Her run farklı bir yol izler — stokastik süreç!

# 7. Markov Decision Process (MDP — Markov Karar Süreci)

MDP, Markov Zinciri'ne **aksiyonlar** ve **ödüller** ekler:

```
MDP = (S, A, T, R, γ)
```

- **S**: Durum uzayı (State Space)
- **A**: Aksiyon uzayı (Action Space)
- **T(s, a, s')**: Geçiş olasılığı — s durumunda a aksiyonu ile s' durumuna geçiş olasılığı
- **R(s, a, s')**: Ödül fonksiyonu — s durumunda a yapıp s' durumuna geçince alınan ödül
- **γ**: İskonto faktörü

## Örnek MDP

3 durum (s0, s1, s2) ve 3 aksiyon (a0, a1, a2) var:
- Bazı aksiyonlar bazı durumlarda mümkün değil (`None`)
- Hedef: Her durumda en iyi aksiyonu öğrenmek


In [ ]:
# MDP tanımlaması

# Geçiş olasılıkları: transition_probabilities[s][a][s']
# None: Bu durum-aksiyon kombinasyonu mümkün değil
transition_probabilities = [
    # s0: a0, a1, a2 aksiyonları mümkün
    [[0.7, 0.3, 0.0], [1.0, 0.0, 0.0], [0.8, 0.2, 0.0]],
    # s1: a0 ve a2 mümkün, a1 yok (None)
    [[0.0, 1.0, 0.0], None, [0.0, 0.0, 1.0]],
    # s2: sadece a1 mümkün, a0 ve a2 yok
    [None, [0.8, 0.1, 0.1], None]
]

# Ödüller: rewards[s][a][s']
# rewards[s][a][s'] = s durumunda a yapıp s' durumuna geçince alınan ödül
rewards = [
    # s0:
    [[+10, 0, 0], [0, 0, 0], [0, 0, 0]],  # s0→s0 geçişi +10 ödül veriyor
    # s1:
    [[0, 0, 0], [0, 0, 0], [0, 0, -50]],   # s1→s2 geçişi -50 ceza
    # s2:
    [[0, 0, 0], [+40, 0, 0], [0, 0, 0]]    # s2→s0 geçişi +40 ödül
]

# Her durumda hangi aksiyonlar mümkün?
possible_actions = [[0, 1, 2], [0, 2], [1]]
# s0: a0, a1, a2 — tüm aksiyonlar
# s1: a0, a2 — a1 mümkün değil
# s2: a1 — sadece a1

# 8. Q-Value Iteration (Q-Değer İterasyonu)

## Q-Value (Eylem-Değer Fonksiyonu)

`Q*(s, a)`: s durumunda a aksiyonunu yapıp sonra **optimal politikayı** izlersek beklenen diskontlanmış toplam ödül.

## Bellman Optimallik Denklemi

```
Q*(s, a) = Σ_{s'} T(s,a,s') * [R(s,a,s') + γ * max_{a'} Q*(s', a')]
```

Bu denklem iteratif olarak çözülebilir:
1. Tüm Q değerlerini 0 ile başlat
2. Her adımda Bellman denklemini uygula
3. Q değerleri yakınsamaya kadar tekrarla

## Optimal Politika

```
π*(s) = argmax_a Q*(s, a)
```

En yüksek Q değerini veren aksiyonu seç.


In [ ]:
# Q-value tablosunu başlat
# Q_values[s, a] = s durumunda a aksiyonunu yapmanın Q değeri
Q_values = np.full((3, 3), -np.inf)  # İmkânsız aksiyonlar için -∞

# Mümkün olan aksiyonlar için 0 ile başlat
for state, actions in enumerate(possible_actions):
    Q_values[state, actions] = 0.0  # Başlangıç: tüm Q değerleri 0

print("Başlangıç Q tablosu:")
print(Q_values)

In [ ]:
gamma = 0.90  # İskonto faktörü γ

history1 = []  # Grafik için Q değer geçmişini sakla

for iteration in range(50):
    Q_prev = Q_values.copy()   # Önceki iterasyonun Q değerlerini sakla
    history1.append(Q_prev)    # Geçmişe ekle
    
    for s in range(3):                    # Her durum için
        for a in possible_actions[s]:     # Her mümkün aksiyon için
            # Bellman Optimallik Denklemi:
            # Q(s,a) = Σ_{s'} T(s,a,s') * [R(s,a,s') + γ * max_{a'} Q(s',a')]
            Q_values[s, a] = np.sum([
                transition_probabilities[s][a][sp]    # Geçiş olasılığı
                * (rewards[s][a][sp]                  # Anlık ödül
                   + gamma * Q_prev[sp].max())         # Diskontlu gelecek değeri
                for sp in range(3)                    # Tüm sonraki durumlar
            ])

history1 = np.array(history1)  # Numpy array'e çevir

In [ ]:
# Yakışmış Q değerleri
print("Q-Value Tablosu (50 iterasyon sonrası):")
Q_values
# Her satır bir durum, her sütun bir aksiyon
# -inf: İmkânsız aksiyonlar
# Yüksek değer: İyi aksiyon (yüksek beklenen kümülatif ödül)

In [ ]:
# Her durum için optimal aksiyon (en yüksek Q değerine sahip aksiyon)
Q_values.argmax(axis=1)
# s0 → a0 (en iyi aksiyon s0'da a0: yüksek Q değeri)
# s1 → a0
# s2 → a1 (zaten tek seçenek)

# 9. Q-Learning

## Q-Value Iteration vs Q-Learning

| Özellik | Q-Value Iteration | Q-Learning |
|---------|-------------------|------------|
| Geçiş modeli | **Gerekli** (T(s,a,s') biliniyor) | **Gerekmez** (model-free) |
| Ortam | Tam bilinir | Sadece gözlemleme yeterli |
| Yöntem | Dinamik programlama | Stokastik güncelleme |

## Q-Learning Güncellemesi (Temporal Difference)

```
Q(s,a) ← Q(s,a) + α * [r + γ * max_{a'} Q(s',a') - Q(s,a)]
```

- `r + γ * max Q(s',a')`: TD hedefi (Bellman denklemi tahmini)
- `Q(s,a)`: Mevcut tahmin
- `α`: Öğrenme hızı
- Parantez içi: TD hatası

## Öğrenme Hızı Azalması (Learning Rate Decay)

`α(t) = α₀ / (1 + t * decay)`

İterasyonlar ilerledikçe α küçülür → erken keşif, sonra istikrar.


In [ ]:
def step(state, action):
    """
    MDP'de bir adım simüle eder (geçiş olasılıklarına göre rastgele).
    
    Q-Learning model-free olduğu için bu fonksiyon normalde bilinmez.
    Burada eğitim amacıyla simulasyon yapıyoruz.
    
    Args:
        state: Mevcut durum (0, 1 veya 2)
        action: Gerçekleştirilecek aksiyon
    
    Returns:
        next_state: Sonraki durum (stokastik)
        reward: Alınan anlık ödül
    """
    probas     = transition_probabilities[state][action]  # Geçiş olasılıkları
    next_state = np.random.choice([0, 1, 2], p=probas)   # Rastgele sonraki durum
    reward     = rewards[state][action][next_state]       # Anlık ödül
    return next_state, reward

In [ ]:
def exploration_policy(state):
    """
    Keşif politikası: Mevcut durumda mümkün olan aksiyonlardan rastgele birini seç.
    
    Q-Learning'in çalışması için tüm durum-aksiyon çiftleri yeterince ziyaret edilmeli.
    Tamamen rastgele keşif (epsilon=1) bu garantiyi sağlar.
    
    Gerçek DQN'de epsilon-greedy kullanılır (zamanla epsilon düşer).
    """
    return np.random.choice(possible_actions[state])

In [ ]:
# Q tablosunu sıfırdan başlat (Q-Value Iteration ile aynı başlangıç)
np.random.seed(42)
Q_values = np.full((3, 3), -np.inf)
for state, actions in enumerate(possible_actions):
    Q_values[state][actions] = 0

In [ ]:
# Q-Learning hiper parametreleri
alpha0 = 0.05   # Başlangıç öğrenme hızı
decay  = 0.005  # Öğrenme hızı azalma katsayısı
gamma  = 0.90   # İskonto faktörü
state  = 0      # Başlangıç durumu: s0
history2 = []   # Grafik için Q değer geçmişi

# 10.000 iterasyon Q-Learning döngüsü
for iteration in range(10_000):
    history2.append(Q_values.copy())  # Mevcut Q tablosunu kaydet
    
    # Keşif politikasıyla aksiyon seç (tamamen rastgele)
    action = exploration_policy(state)
    
    # Aksiyonu uygula ve gözlem al
    next_state, reward = step(state, action)
    
    # Sonraki durumdaki maksimum Q değeri (greedy gelecek tahmin)
    next_value = Q_values[next_state].max()
    
    # Azalan öğrenme hızı: Erken iterasyonlarda büyük, sonra küçülür
    alpha = alpha0 / (1 + iteration * decay)
    
    # Q-Learning TD güncellemesi:
    # Q(s,a) ← (1-α)*Q(s,a) + α*(r + γ*max Q(s',a'))
    # Eşdeğer: Q(s,a) += α * [r + γ*max Q(s',a') - Q(s,a)]
    Q_values[state, action] *= 1 - alpha                         # Eski değeri azalt
    Q_values[state, action] += alpha * (reward + gamma * next_value)  # TD hedefini ekle
    
    state = next_state  # Bir sonraki duruma geç

history2 = np.array(history2)

In [ ]:
# Q-Value Iteration vs Q-Learning karşılaştırma grafiği
# Her iki yöntem de s0'da a0 için aynı Q değerine yakınsamalı

true_Q_value = history1[-1, 0, 0]  # Gerçek Q*(s0, a0) değeri

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

axes[0].set_ylabel("Q-value$(s_0, a_0)$", fontsize=14)
axes[0].set_title("Q-value iteration", fontsize=14)
axes[1].set_title("Q-learning", fontsize=14)

for ax, width, history in zip(axes, (50, 10000), (history1, history2)):
    ax.plot([0, width], [true_Q_value, true_Q_value], "k--",
            label="Gerçek Q* değeri")          # Yatay kesikli çizgi
    ax.plot(np.arange(width), history[:, 0, 0], "b-", linewidth=2,
            label="Tahmin edilen Q değeri")    # Öğrenme eğrisi
    ax.set_xlabel("Iterations", fontsize=14)
    ax.axis([0, width, 0, 24])
    ax.grid(True)

plt.show()
# Sol: Q-Value Iteration — hızlı yakınsar (50 adım)
# Sağ: Q-Learning — daha yavaş yakınsar (1000+ adım) ama model gerektirmez

# 10. Deep Q-Network (DQN — Derin Q-Ağı)

## Neden Derin Ağ?

Tablolu Q-Learning büyük/sürekli durum uzaylarında çalışmaz (milyonlarca durum).  
**DQN**, Q tablosunu bir sinir ağıyla yaklaştırır:

```
Q_θ(s, a) ≈ Q*(s, a)   ∀a
```

Sinir ağı tüm aksiyonlar için Q değerlerini **aynı anda** hesaplar.

## DQN Önemli Bileşenleri

### 1. Replay Buffer (Deneyim Tekrar Tamponu)
- Geçmiş deneyimleri `(s, a, r, s', done)` formatında saklar
- Rastgele mini-batch örnekleme → korelasyonları kırar
- Veri verimliliğini artırır (aynı deneyimi birçok kez kullan)

### 2. ε-Greedy Keşif (Epsilon-Greedy Exploration)
- ε olasılığıyla **rastgele** aksiyon (keşif — exploration)
- (1-ε) olasılığıyla **en iyi** aksiyon (sömürü — exploitation)
- Eğitim boyunca ε azalır: 1.0 → 0.01

### 3. DQN Loss Fonksiyonu

```
Loss = MSE(Q_θ(s,a), r + γ * max_{a'} Q_θ(s',a'))
```

Hedef: Bellman denklemini sağlayan Q değerleri bul.


In [ ]:
class DQN(nn.Module):
    """
    CartPole için Deep Q-Network.
    
    Giriş: 4 boyutlu gözlem [cart_pos, cart_vel, pole_angle, pole_angular_vel]
    Çıkış: 2 boyutlu Q değerleri [Q(s, left), Q(s, right)]
    
    Her iki aksiyon için Q değeri aynı anda hesaplanır (verimli).
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 32), nn.ReLU(),   # 4 giriş → 32 gizli nöron
            nn.Linear(32, 32), nn.ReLU(),  # 32 → 32 (daha derin ağ)
            nn.Linear(32, 2)               # 32 → 2 Q değeri (sol ve sağ)
        )

    def forward(self, state):
        return self.net(state)  # [B, 4] → [B, 2]

In [ ]:
def choose_dqn_action(model, obs, epsilon=0.0):
    """
    ε-Greedy aksiyon seçimi.
    
    Eğitim sırasında keşif (exploration) için yüksek epsilon kullanılır.
    Değerlendirme sırasında epsilon=0.0 (tamamen greedy/açgözlü).
    
    Args:
        model: DQN modeli
        obs: Mevcut gözlem
        epsilon: Keşif olasılığı [0, 1]
    
    Returns:
        Seçilen aksiyonun indeksi (0 veya 1)
    """
    if torch.rand(()) < epsilon:
        # Epsilon olasılığıyla RASTGELE aksiyon (keşif)
        # torch.rand(()): 0-1 arası tek bir skaler rastgele sayı
        return torch.randint(2, size=()).item()
    else:
        # (1-epsilon) olasılığıyla EN İYİ aksiyon (greedy)
        state    = torch.as_tensor(obs)
        Q_values = model(state)              # Her aksiyon için Q değerleri
        return Q_values.argmax().item()      # En yüksek Q değerli aksiyonu seç

In [ ]:
def sample_experiences(replay_buffer, batch_size):
    """
    Replay buffer'dan rastgele mini-batch örnekler.
    
    Neden rastgele örnekleme?
    - Ardışık deneyimler yüksek korelasyonlu → kararlı olmayan eğitim
    - Rastgele örnekleme korelasyonu kırar → daha kararlı gradient
    
    Returns:
        6 elemanlı tuple: (state, action, reward, next_state, done, truncated)
        Her eleman [batch_size, ...] boyutlu tensor
    """
    # Rastgele batch_size indeks seç
    indices = torch.randint(len(replay_buffer), size=[batch_size])
    batch   = [replay_buffer[index] for index in indices.tolist()]
    
    # Her deneyim tuple'ını ayrı tensor'lara dönüştür
    return [to_tensor([exp[index] for exp in batch]) for index in range(6)]


def to_tensor(data):
    """
    Python listesini numpy array üzerinden PyTorch tensor'a çevirir.
    float64 → float32 dönüşümü otomatik yapılır (GPU optimizasyonu için).
    """
    array = np.stack(data)
    # Gymnasium float64 kullanır; PyTorch genellikle float32 kullanır
    dtype = torch.float32 if array.dtype == np.float64 else None
    return torch.as_tensor(array, dtype=dtype)

## 10.1 Circular Buffer (Dairesel Tampon)

Büyük replay buffer'lar için daha verimli bir implementasyon. Maksimum kapasiteye ulaşıldığında eski deneyimlerin üzerine yazar.

In [ ]:
class ReplayBuffer:
    """
    Sabit boyutlu dairesel tampon (circular buffer) implementasyonu.
    
    Özellikler:
    - Maksimum kapasiteye ulaşıldığında en eski veriyi siler (FIFO)
    - O(1) ekleme ve erişim
    - `deque(maxlen=...)` yerine daha bellek-verimli
    
    Örnek: max_length=4 için
    - [0, 1, 2, 3] dolu
    - 4 eklenir → [4, 1, 2, 3] (index=0 üzerine yazılır)
    - 5 eklenir → [4, 5, 2, 3] (index=1 üzerine yazılır)
    """
    def __init__(self, max_length):
        self.data       = [None] * max_length  # Sabit boyutlu liste
        self.max_length = max_length
        self.index      = 0    # Bir sonraki yazma konumu
        self.length     = 0    # Mevcut eleman sayısı

    def append(self, obj):
        """Tampona yeni eleman ekle (doluysa en eskinin üzerine yaz)."""
        self.data[self.index] = obj
        # Uzunluğu max_length ile sınırla
        self.length = min(self.length + 1, self.max_length)
        # Dairesel: index sonuna gelince başa dön
        self.index  = (self.index + 1) % self.max_length

    def __getitem__(self, index):
        if index < 0 or index >= self.length:
            raise IndexError(f"replay buffer index out of range: {index}")
        return self.data[index]

    def __len__(self):
        return self.length

# Dairesel buffer davranışını test et
buff = ReplayBuffer(max_length=4)
for i in range(8):
    buff.append(i)
    print(f"Length: {len(buff)}, data: {buff.data}")
# 0-3 arası: Buffer doluyor
# 4-7 arası: Eski veriler üzerine yazılıyor

In [ ]:
def play_and_record_episode(model, env, replay_buffer, epsilon, seed=None):
    """
    Bir episode oynar ve tüm deneyimleri replay buffer'a kaydeder.
    
    Her adımda kaydedilen deneyim: (obs, action, reward, next_obs, done, truncated)
    
    Args:
        model: DQN modeli (aksiyon seçimi için)
        env: Gymnasium ortamı
        replay_buffer: Deneyimlerin saklanacağı buffer
        epsilon: ε-greedy keşif oranı
        seed: Tekrarlanabilirlik seed'i
    
    Returns:
        total_rewards: Episode boyunca toplanan toplam ödül
    """
    obs, _info = env.reset(seed=seed)
    total_rewards = 0
    model.eval()              # Değerlendirme modu
    
    with torch.no_grad():     # Gradient hesaplamayı kapat (sadece aksiyon seçimi)
        while True:
            action    = choose_dqn_action(model, obs, epsilon)  # ε-greedy
            next_obs, reward, done, truncated, _info = env.step(action)
            
            # Deneyimi (s, a, r, s', done, truncated) formatında kaydet
            experience = (obs, action, reward, next_obs, done, truncated)
            replay_buffer.append(experience)
            
            total_rewards += reward
            
            if done or truncated:
                return total_rewards
            obs = next_obs  # Bir sonraki adım için gözlemi güncelle

In [ ]:
def dqn_training_step(model, optimizer, criterion, replay_buffer, batch_size,
                      discount_factor):
    """
    Replay buffer'dan bir mini-batch örnekler ve DQN'i bir adım günceller.
    
    DQN Loss:
    target = r + γ * max_{a'} Q(s', a')   [s' terminal değilse]
    target = r                              [s' terminal ise]
    
    Loss = MSE(Q(s, a), target)
    
    Önemli: next_Q_value hesaplanırken gradient OLMAMALI (target sabit tutulmalı).
    torch.inference_mode() bunu sağlar.
    """
    # Replay buffer'dan rastgele deneyim batch'i al
    experiences = sample_experiences(replay_buffer, batch_size)
    state, action, reward, next_state, done, truncated = experiences
    
    # Sonraki durumun Q değerlerini hesapla (gradient olmadan — target sabit)
    with torch.inference_mode():
        next_Q_value = model(next_state)  # [batch_size, 2]
    
    # Her sonraki durum için maksimum Q değerini al
    max_next_Q_value, _ = next_Q_value.max(dim=1)  # [batch_size]
    
    # Terminal durumlar için Q değeri 0 (oyun bitti, gelecek ödül yok)
    # done OR truncated: İkisi de episode'un bittiğini gösterir
    running = (~(done | truncated)).float()  # 1: devam ediyor, 0: bitti
    
    # Bellman hedef değeri:
    # terminal durumda: target = r
    # terminal olmayan: target = r + γ * max Q(s', a')
    target_Q_value = reward + running * discount_factor * max_next_Q_value
    
    # Mevcut durumun tüm Q değerlerini hesapla
    all_Q_values = model(state)  # [batch_size, 2]
    
    # Sadece seçilen aksiyonun Q değerini al
    # action.unsqueeze(1): [batch_size] → [batch_size, 1]
    # gather: Her satırda belirtilen indeksteki değeri al
    Q_value = all_Q_values.gather(dim=1, index=action.unsqueeze(1))  # [batch_size, 1]
    
    # MSE loss hesapla ve backpropagation yap
    loss = criterion(Q_value, target_Q_value.unsqueeze(1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
from collections import deque

def train_dqn(model, env, replay_buffer, optimizer, criterion, n_episodes=800,
              warmup=30, batch_size=32, discount_factor=0.95):
    """
    DQN eğitim döngüsü.
    
    Eğitim akışı:
    1. epsilon'u mevcut episode sayısına göre hesapla (başta 1.0, sonda 0.01)
    2. Bir episode oyna ve deneyimleri replay buffer'a kaydet
    3. warmup episode'dan sonra her adımda bir training step yap
    
    Args:
        model: DQN modeli
        env: Gymnasium ortamı
        replay_buffer: Deneyim tamponu
        optimizer: Adam gibi optimizer
        criterion: MSE gibi kayıp fonksiyonu
        n_episodes: Toplam episode sayısı
        warmup: İlk kaç episode'da sadece veri toplanacak (eğitim yok)
        batch_size: Mini-batch boyutu
        discount_factor: γ
    
    Returns:
        totals: Her episode'un toplam ödülü
    """
    totals = []
    
    for episode in range(n_episodes):
        # Epsilon schedule: 1.0'dan 0.01'e doğrusal düşüş
        # İlk 500 episode: exploration ağırlıklı
        # 500+ episode: exploitation ağırlıklı (min 0.01 keşif)
        epsilon = max(1 - episode / 500, 0.01)
        
        seed = torch.randint(0, 2**32, size=()).item()
        
        # Episode oyna ve deneyimleri buffer'a kaydet
        total_rewards = play_and_record_episode(model, env, replay_buffer,
                                                epsilon, seed=seed)
        print(f"\rEpisode: {episode + 1}, Rewards: {total_rewards}", end=" ")
        totals.append(total_rewards)
        
        # Warmup: İlk 30 episode'da sadece veri topla, eğitim yapma
        # Buffer yeterince dolduğunda eğitime başla
        if episode >= warmup:
            dqn_training_step(model, optimizer, criterion, replay_buffer,
                              batch_size, discount_factor)
    
    return totals

# DQN modelini oluştur ve eğit
torch.manual_seed(42)
dqn     = DQN()
optimizer = torch.optim.NAdam(dqn.parameters(), lr=0.03)
mse     = nn.MSELoss()

# deque(maxlen=100_000): 100.000 deneyim saklayan otomatik döngüsel liste
replay_buffer = deque(maxlen=100_000)

totals = train_dqn(dqn, env, replay_buffer, optimizer, mse)

In [ ]:
# Eğitim boyunca episode ödülleri grafiği
plt.figure(figsize=(8, 4))
plt.plot(totals)
plt.xlabel("Episode", fontsize=14)
plt.ylabel("Sum of rewards", fontsize=14)
plt.grid(True)
plt.show()
# Beklenti: Başlarda düşük (rastgele keşif), giderek artar
# İyi bir DQN tutarlı olarak 1000 adım ulaşabilmeli

In [ ]:
# Eğitilmiş DQN'in animasyonunu göster
torch.manual_seed(42)
dqn.eval()

def dqn_policy(obs):
    """DQN modelini kullanan politika wrapper fonksiyonu."""
    with torch.no_grad():
        return choose_dqn_action(dqn, obs)  # epsilon=0: tamamen greedy

show_one_episode(dqn_policy)
# Sonuç: Çok kararlı bir CartPole oynaması beklenmeli!

# 11. Actor-Critic (Aktör-Eleştirmen)

## REINFORCE'un Sınırlamaları

REINFORCE'da:
- Episode bitene kadar return hesaplanamaz → yüksek varyans
- Tüm return tahmini Monte Carlo → gürültülü gradientler

**Actor-Critic**, bu sorunu **Critic** (eleştirmen) ekleyerek çözer.

## Actor-Critic Mimarisi

```
          ┌─────────────┐
          │   Paylaşılan │
          │   Gövde     │ ← Ortak özellik çıkarıcı
          └──────┬──────┘
                 │
         ┌───────┴───────┐
         ▼               ▼
    [Actor Head]    [Critic Head]
    Aksiyon logiti  Durum değeri V(s)
```

**Actor (Aktör)**: Politikayı temsil eder — hangi aksiyonu seç?  
**Critic (Eleştirmen)**: Durum değerini tahmin eder — mevcut durum ne kadar iyi?

## TD Error (Temporal Difference Hatası)

```
δ_t = r_t + γ * V(s_{t+1}) - V(s_t)
```

- δ > 0: Beklentiden daha iyi → Bu aksiyonun olasılığını artır
- δ < 0: Beklentiden daha kötü → Bu aksiyonun olasılığını azalt

## Loss Fonksiyonları

```
Actor Loss  = -log π(a|s) * δ  (gradient yükselt)
Critic Loss = MSE(V(s), r + γ*V(s'))
Total Loss  = Actor Loss + w * Critic Loss
```


In [ ]:
class ActorCritic(nn.Module):
    """
    Actor-Critic modeli.
    
    Actor ve Critic paylaşılan bir gövde kullanır.
    Bu, ortak özellik temsilini verimli öğrenmelerini sağlar.
    
    Giriş: 4 boyutlu gözlem
    Çıkış: (aksiyon logiti, durum değeri V(s))
    """
    def __init__(self):
        super().__init__()
        
        # Paylaşılan gövde: Hem Actor hem Critic bu katmanları kullanır
        # Düşük seviye özellikleri burada öğrenilir
        self.body = nn.Sequential(
            nn.Linear(4, 32), nn.ReLU(),   # 4 → 32
            nn.Linear(32, 32), nn.ReLU()   # 32 → 32
        )
        
        # Actor head: Gövdeden gelen özelliklerden aksiyon logiti üretir
        # Logit → Bernoulli dağılımı → aksiyon örneklemesi
        self.actor_head = nn.Linear(32, 1)
        
        # Critic head: Gövdeden gelen özelliklerden durum değeri V(s) üretir
        # V(s): Bu durumda beklenen diskontlanmış toplam ödül
        self.critic_head = nn.Linear(32, 1)

    def forward(self, state):
        """
        Returns:
            actor_logit: Aksiyon logiti (Bernoulli için)
            state_value: V(s) — durum değeri tahmini
        """
        features = self.body(state)                 # Paylaşılan özellikler
        return self.actor_head(features), self.critic_head(features)

In [ ]:
def choose_action_and_evaluate(model, obs):
    """
    Actor-Critic modelinden aksiyon seçer ve değerlendirme yapar.
    
    REINFORCE'taki choose_action'dan farkı:
    Sadece log_prob değil, ayrıca state_value de döndürür.
    
    Returns:
        action: 0 veya 1 (int)
        log_prob: log π(a|s) — Actor loss için
        state_value: V(s) — Critic loss için
    """
    state  = torch.as_tensor(obs)
    logit, state_value = model(state)              # Actor ve Critic çıkışları
    
    # Bernoulli dağılımından aksiyon örnekle
    dist     = torch.distributions.Bernoulli(logits=logit)
    action   = dist.sample()
    log_prob = dist.log_prob(action)
    
    return int(action.item()), log_prob, state_value

In [ ]:
def ac_training_step(optimizer, criterion, state_value, target_value, log_prob,
                     critic_weight):
    """
    Actor-Critic güncelleme adımı.
    
    TD hatası: δ = target_value - state_value
    
    Actor Loss: -log π(a|s) * δ
    - δ > 0 (tahmin çok düşük): Bu aksiyonu daha sık seç
    - δ < 0 (tahmin çok yüksek): Bu aksiyonu daha az seç
    
    Critic Loss: MSE(V(s), r + γ*V(s'))
    - Critic'i TD hedefine yaklaştır
    
    Önemli: td_error.detach()
    - TD hatası Actor loss için sabit tutulmalı
    - Aksi halde gradient Critic üzerinden de Actor'a akar → kararsız eğitim
    
    Args:
        state_value: Critic'in mevcut durum tahmini V(s)
        target_value: TD hedefi r + γ*V(s')
        log_prob: Actor'ın log π(a|s) değeri
        critic_weight: Critic loss'un toplam loss'taki ağırlığı
    """
    td_error = target_value - state_value          # δ = hedef - tahmin
    
    # Actor loss: -log π(a|s) * δ
    # td_error.detach(): Gradient'in Critic'e akmasını engelle
    actor_loss = -log_prob * td_error.detach()
    
    # Critic loss: MSE(V(s), hedef)
    critic_loss = criterion(state_value, target_value)
    
    # Toplam loss
    loss = actor_loss + critic_weight * critic_loss
    
    optimizer.zero_grad()
    loss.backward()   # Hem Actor hem Critic için gradient hesapla
    optimizer.step()

In [ ]:
def get_target_value(model, next_obs, reward, done, truncated, discount_factor):
    """
    TD hedef değerini hesaplar.
    
    Terminal durumda (done/truncated): target = r
    Terminal olmayan durumda: target = r + γ * V(s')
    
    torch.inference_mode(): Gradient hesaplamadan hızlı forward pass
    """
    with torch.inference_mode():
        # Sonraki durum için Critic değerlendirmesi (gradient olmadan)
        _, _, next_state_value = choose_action_and_evaluate(model, next_obs)
    
    # Terminal durumda gelecek ödül yok
    running      = 0.0 if (done or truncated) else 1.0
    target_value = reward + running * discount_factor * next_state_value
    return target_value

In [ ]:
def run_episode_and_train(model, optimizer, criterion, env, discount_factor,
                          critic_weight, seed=None):
    """
    Bir episode boyunca her adımda online (anlık) güncelleme yapar.
    
    REINFORCE'tan farkı: Episode bitmesini beklemeden her adımda güncelleme.
    Bu, daha düşük varyans sağlar.
    
    Returns:
        total_rewards: Episode toplam ödülü
    """
    obs, _info = env.reset(seed=seed)
    total_rewards = 0
    
    while True:
        # Aksiyon seç ve değerlendir (Actor + Critic)
        action, log_prob, state_value = choose_action_and_evaluate(model, obs)
        
        # Aksiyonu uygula
        next_obs, reward, done, truncated, _info = env.step(action)
        
        # TD hedef değerini hesapla
        target_value = get_target_value(model, next_obs, reward, done,
                                        truncated, discount_factor)
        
        # Actor ve Critic'i güncelle (her adımda!)
        ac_training_step(optimizer, criterion, state_value, target_value,
                         log_prob, critic_weight)
        
        total_rewards += reward
        
        if done or truncated:
            return total_rewards
        
        obs = next_obs  # Sonraki adım için gözlemi güncelle

In [ ]:
def train_actor_critic(model, optimizer, criterion, env, n_episodes=400,
                       discount_factor=0.95, critic_weight=0.3):
    """
    Actor-Critic eğitim döngüsü.
    
    Args:
        critic_weight: Critic loss'un ağırlığı (0.3: Critic üçte bir katkı)
        discount_factor: γ = 0.95 (gelecek ödülleri hafifçe iskonto et)
    """
    totals = []
    model.train()  # Eğitim modu
    
    for episode in range(n_episodes):
        seed = torch.randint(0, 2**32, size=()).item()
        total_rewards = run_episode_and_train(model, optimizer, criterion, env,
                                              discount_factor, critic_weight,
                                              seed=seed)
        totals.append(total_rewards)
        print(f"\rEpisode: {episode + 1}, Rewards: {total_rewards}", end=" ")
    
    return totals

In [ ]:
# Actor-Critic modelini eğit
torch.manual_seed(42)
ac_model  = ActorCritic()
optimizer = torch.optim.NAdam(ac_model.parameters(), lr=1.1e-3)
criterion = nn.MSELoss()

# 400 episode, gamma=0.95, critic_weight=0.3
totals = train_actor_critic(ac_model, optimizer, criterion, env)

In [ ]:
# Eğitilmiş Actor-Critic'in animasyonunu göster
torch.manual_seed(42)
ac_model.eval()

def actor_critic_policy(obs):
    """Actor-Critic modelini kullanan politika wrapper fonksiyonu."""
    with torch.no_grad():
        action, _, _ = choose_action_and_evaluate(ac_model, obs)
        return action

show_one_episode(actor_critic_policy)

In [ ]:
# Eğitim boyunca episode ödülleri grafiği
plt.figure(figsize=(8, 4))
plt.plot(totals)
plt.xlabel("Episode", fontsize=14)
plt.ylabel("Sum of rewards", fontsize=14)
plt.grid(True)
plt.show()
# Actor-Critic REINFORCE'tan genellikle daha hızlı öğrenir (daha düşük varyans)

# 12. PPO (Proximal Policy Optimization) — Atari Breakout

## PPO Nedir?

**PPO**, modern RL'nin en popüler algoritmalarından biridir (OpenAI, DeepMind vb. kullanır).

## PPO'nun Temel Fikri

Policy Gradient yöntemlerinde büyük güncellemeler politikayı bozabilir.  
PPO, güncellemeleri sınırlamak için **clipping** kullanır:

```
L^CLIP = E_t [min(r_t * A_t, clip(r_t, 1-ε, 1+ε) * A_t)]
```

- `r_t = π_θ(a|s) / π_θ_old(a|s)`: Yeni/eski politika oranı
- `A_t`: Advantage (Avantaj) — ne kadar beklenenden iyi?
- `clip(r_t, 1-ε, 1+ε)`: Oranı dar bir aralıkta tut

Bu sayede her güncelleme politikayı çok fazla değiştirmez.

## Stable-Baselines3

Manuel implementasyon yerine **stable_baselines3** kütüphanesini kullanıyoruz.  
Atari oyunları derin CNN politikaları gerektirir.

## Atari Ön İşleme

Stable-Baselines3 otomatik olarak uygular:
- Gri tonlamalı dönüşüm: RGB(210×160×3) → Gri(84×84×1)
- Frame stacking: Son 4 frame birleştirilir → hareket bilgisi
- Frame skipping: Her 4. frame işlenir → hız


In [ ]:
import ale_py  # Atari Learning Environment — Atari oyun simulatörü

# ALE arayüzünü başlat
# Atari ROM'larını yükler ve Gymnasium ile entegrasyon sağlar
ale = ale_py.ALEInterface()

In [ ]:
# Anaconda ile kurulduysa ROM'ların yüklenmesi gerekebilir
# Normal pip kurulumunda bu adım gerekli değildir
gymnasium_atari_was_installed_using_anaconda = False

if gymnasium_atari_was_installed_using_anaconda:
    %pip install -qU AutoROM
    from pathlib import Path
    from AutoROM import main as autorom
    # Atari lisansını kabul ederek ROM'ları kur
    autorom(accept_license=True, source_file=None,
            install_dir=Path(ale_py.roms.__path__[0]), quiet=False)

In [ ]:
from stable_baselines3.common.env_util import make_atari_env

# Paralel Atari ortamları oluştur (4 bağımsız ortam)
# Vektörleştirilmiş ortamlar: Her adımda 4 farklı oyun adımı → daha verimli eğitim
# BreakoutNoFrameskip-v4: Frame atlamayı manuel kontrol etmek için
envs = make_atari_env("BreakoutNoFrameskip-v4", n_envs=4, seed=42)

# obs: 4 ortamın ilk gözlemi — shape: (n_envs, H, W, C) = (4, 84, 84, 1)
obs = envs.reset()

In [ ]:
# Gözlem boyutunu kontrol et
# (4, 84, 84, 1): 4 paralel ortam, 84×84 gri tonlamalı görüntü, 1 kanal
obs.shape

In [ ]:
# Orijinal ve ön işlenmiş görüntüleri karşılaştır
plt.figure(figsize=(9, 4))

# Orijinal görüntü: 210×160 RGB (tam çözünürlük, renkli)
plt.subplot(1, 2, 1)
plt.title("Original 210 ✕ 160 RGB")
plt.imshow(envs.get_images()[0])  # İlk ortamın orijinal görüntüsü
plt.axis("off")

# Ön işlenmiş görüntü: 84×84 Gri tonlamalı
plt.subplot(1, 2, 2)
plt.title("Preprocessed 84 ✕ 84 Grayscale")
plt.imshow(obs[0, :, :, 0], cmap="gray")  # İlk ortamın işlenmiş görüntüsü
plt.axis("off")

plt.show()
# Stable-Baselines3 otomatik:
# 1. Gri tonlamaya çevirir (renk bilgisi RL için çok önemli değil)
# 2. 84×84 boyutuna küçültür (hesaplama verimliliği)

In [ ]:
from stable_baselines3.common.vec_env import VecFrameStack

# Frame Stacking: Son 4 frame'i birleştir
# Neden? Tek bir frame hareket yönünü göstermez
# 4 frame yığını: Topun/karakterin hareketini görmemizi sağlar
# n_stack=4: Son 4 frame → obs shape: (4, 84, 84, 4)
envs_stacked = VecFrameStack(envs, n_stack=4)
obs = envs_stacked.reset()  # (4, 84, 84, 4): 4 ortam, 84×84, 4 frame yığını

In [ ]:
from stable_baselines3 import PPO

# TensorBoard log dizini
tensorboard_logdir = "my_ppo_breakout_tensorboard"

# PPO modelini oluştur
ppo_model = PPO(
    "CnnPolicy",           # Politika: CNN tabanlı (görüntü giriş için)
    envs_stacked,          # Vektörleştirilmiş + frame stacked ortam
    device=device,         # GPU/CPU
    learning_rate=2.5e-4,  # Adam optimizer learning rate
    batch_size=256,        # Mini-batch boyutu
    n_steps=256,           # Her güncelleme öncesi her ortamda atılan adım
    n_epochs=4,            # Her veri setini kaç kez kullan
    clip_range=0.1,        # PPO clipping parametresi ε (politika güncellemesini sınırlar)
    vf_coef=0.5,           # Value function loss katsayısı
    ent_coef=0.01,         # Entropy bonus katsayısı (keşfetmeyi teşvik eder)
    gamma=0.99,            # İskonto faktörü (uzun vadeli oyun için yüksek)
    verbose=0,             # Sessiz mod (TensorBoard kullanacağız)
    tensorboard_log=tensorboard_logdir  # Metrikleri TensorBoard'a kaydet
)

In [ ]:
# PPO modelinin politika ağı mimarisini göster
# CnnPolicy: NatureCNN (3 katman Conv) + 2 katman FC
ppo_model.policy

In [ ]:
# TensorBoard'u başlat (eğitim sırasında gerçek zamanlı izleme)
# Tarayıcıda localhost:6006 adresini açın
%tensorboard --logdir={tensorboard_logdir} --port 6006

In [ ]:
from stable_baselines3.common.callbacks import CheckpointCallback

# Checkpoint callback: Her 100.000 adımda modeli kaydet
# Eğer eğitim durur veya en iyi modeli kurtarmak istersen kullanılır
cb = CheckpointCallback(
    save_freq=100_000,         # Her 100.000 adımda kaydet
    save_path="my_checkpoints",  # Kayıt klasörü
    name_prefix="breakout"     # Dosya adı öneki: breakout_100000_steps.zip vb.
)

# Modeli eğit
# total_timesteps=200_000: Demo için 200K adım (tam eğitim 30M+ adım gerektirir)
# progress_bar=True: Tqdm ilerleme çubuğu
ppo_model.learn(
    total_timesteps=200_000,
    progress_bar=True,
    callback=cb
)

# Eğitilmiş modeli kaydet
ppo_model.save("my_ppo_agent_breakout")

## 12.1 Önceden Eğitilmiş Modeli İndir

30 milyon adım eğitim çok uzun sürer (GPU'da birkaç saat).  
Yazarın önceden eğittiği modeli indirerek kullanabiliriz.


In [ ]:
import urllib.request

# Önceden eğitilmiş modeli GitHub'dan indir
data_root = "https://raw.githubusercontent.com/ageron/data/refs/heads/main"
filename  = "ppo_agent_breakout.zip"

# urllib.request.urlretrieve: URL'den dosya indirir
urllib.request.urlretrieve(f"{data_root}/{filename}", f"my_{filename}")
print(f"Model indirildi: my_{filename}")

In [ ]:
# İndirilen veya en iyi checkpoint'i yükle
ppo_model = PPO.load("my_ppo_agent_breakout")

# Değerlendirme ortamı oluştur (1 ortam, seed=42)
eval_env     = make_atari_env("BreakoutNoFrameskip-v4", n_envs=1, seed=42)
eval_stacked = VecFrameStack(eval_env, n_stack=4)

# Frame'leri kaydet
frames = []
obs    = eval_stacked.reset()

for _ in range(5000):  # En fazla 5000 adım (sonsuz döngüyü önler)
    frames.append(eval_stacked.render())  # Mevcut frame'i kaydet
    
    # deterministic=True: Stokastik değil, en iyi aksiyonu seç (değerlendirme için)
    action, _ = ppo_model.predict(obs, deterministic=True)
    
    obs, reward, done, info = eval_stacked.step(action)
    
    if done[0]:  # İlk ortam bittiyse dur
        break    # Not: VecEnv'de 'truncated' yok, 'done' her ikisini de kapsar

eval_stacked.close()  # Ortam kaynaklarını serbest bırak

In [ ]:
# Breakout oyununun animasyonunu göster
# 30M adım eğitim sonrası ajan tuğlaları kırmayı öğrenmeli
plot_animation(frames)

---

# Özet — Reinforcement Learning Algoritmaları

## Algoritma Karşılaştırması

| Algoritma | Tip | Avantaj | Dezavantaj |
|-----------|-----|---------|------------|
| **Hard-Coded Policy** | Kural tabanlı | Basit, anlaşılır | Genelleme yok |
| **REINFORCE** | Policy Gradient | Basit implementasyon | Yüksek varyans |
| **Q-Value Iteration** | Dinamik Programlama | Kesin çözüm | Model gerektirir |
| **Q-Learning** | Model-Free TD | Model gerekmez | Büyük tablo |
| **DQN** | Deep + Q-Learning | Büyük durum uzayı | Yavaş öğrenme |
| **Actor-Critic** | Policy + Value | Düşük varyans | Karmaşık |
| **PPO** | Policy Gradient | Kararlı, verimli | Hiper-parametre ayarı |

## Anahtar Kavramlar

| Kavram | Açıklama |
|--------|----------|
| **Reward** | Anlık ödül |
| **Return (G_t)** | Diskontlanmış kümülatif ödül |
| **V(s)** | Durum değeri (State Value) |
| **Q(s,a)** | Eylem-değer fonksiyonu (Action-Value) |
| **Policy π** | Gözlem → Eylem eşlemesi |
| **TD Error** | Temporal Difference hatası δ |
| **Replay Buffer** | Geçmiş deneyimleri saklayan tampon |
| **ε-Greedy** | Keşif/sömürü dengesi |
| **Discount Factor γ** | Gelecek ödüllerin iskonto oranı |
| **Advantage A_t** | Beklentiden ne kadar iyi? |
